In [1]:
import pandas as pd
import os
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import joblib



In [2]:
def readDataInPanda(file_paths):
    folder_path = 'data'
    dataframes = []
    for file_path in file_paths: 
        print("read: " + file_path)
        df = pd.read_csv(file_path)
        dataframes.append(df)
        pandas_data = pd.concat(dataframes, ignore_index=True)

    return pandas_data

In [3]:
#Training data were only the Resistance varies the target voltage is set to 90
def getTrainingDataResistance():
    file_paths = ["./data/training/Circuit_ScenarioDataGeneration_Restistance_res.csv"]
    return readDataInPanda(file_paths)

In [4]:
#Training data were the Resistance and the target voltage changes. Values range from 1. to 100. 
def getTrainingDataResistanceAndVoltage():
    file_paths = ["./data/training/Circuit_ScenarioDataGeneration_Volatage_Restistance_res.csv"]
    return readDataInPanda(file_paths)

In [5]:
def getFaultyData():
    folder_path = "./data/faulty"
    file_paths = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.csv')]
    return readDataInPanda(file_paths)

In [6]:
def getFaultlessData():
    folder_path = "./data/faultless"
    file_paths = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.csv')]
    return readDataInPanda(file_paths)

In [7]:
def getMixedData():
    data_frames = getFaultyData()
    data_fame_faltless = getFaultlessData()
    result = pd.concat([data_frames, data_fame_faltless])

    return data_frames

In [8]:
def loadModelAndScalar(model_name):
    scaler = joblib.load("./models/" + model_name + ".save")
    model = tf.keras.models.load_model("./models/" + model_name + ".h5")
    return model, scaler
    

In [9]:
def getAnomalyMask(df_data): 
    anomolous = []
    for idx, row in df_data.iterrows():
        #print(row)
        # Check any of the specified columns for value == 2
        if (row["circ1.bats[1].state"] != 1 or
            row["circ1.bats[2].state"] != 1 or
            row["circ1.bats[3].state"] != 1 or
            row["circ1.s[1].state"]  != 1 or
            row["circ1.s[2].state"]  != 1 or
            row["circ1.s[3].state"]  != 1):
            anomolous.append(True)
        else:
            anomolous.append(False)
    return np.array(anomolous)

In [10]:
# Quick test for anomly mask and check data Integrity
anom_mask = getAnomalyMask(getFaultyData())
for anom in anom_mask:
   assert anom == True
anom_mask = getAnomalyMask(getFaultlessData())
for anom in anom_mask:
   assert anom == False

read: ./data/faulty\Circuit_Scenario1_batery_1_broken_res.csv
read: ./data/faulty\Circuit_Scenario1_switch1_broken_res.csv
read: ./data/faulty\Circuit_Scenario2_battery2_broken_res.csv
read: ./data/faulty\Circuit_Scenario2_battery2_empty_res.csv
read: ./data/faulty\Circuit_Scenario2_switch_2_broken_res.csv
read: ./data/faulty\Circuit_Scenario3_battery_1_empty_res.csv
read: ./data/faulty\Circuit_Scenario3_switch_1_broken_res.csv
read: ./data/faulty\Circuit_Scenario4_battery_1_broken_res.csv
read: ./data/faulty\Circuit_Scenario4_battery_1_empty_res.csv
read: ./data/faulty\Circuit_Scenario4_switch_1_broken_res.csv
read: ./data/faultless\Circuit_Scenario1_res.csv
read: ./data/faultless\Circuit_Scenario2_res.csv
read: ./data/faultless\Circuit_Scenario3_res.csv
read: ./data/faultless\Circuit_Scenario4_res.csv
